In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
#from utils import *
#from utils_cpdir import *
from utils_mvcp import *
from utils import *
from utils_spec import *
import os
from tqdm import tqdm
import json
import os
from datetime import date

In [2]:
# Get the current date
todays_date = date.today()
year = str(todays_date.year)
if todays_date.month < 10:
    month = '0'+str(todays_date.month)
else:
    month = str(todays_date.month)

if todays_date.day < 10:
    day = '0'+str(todays_date.day)
else:
    day = str(todays_date.day)

current_date = year+month+day
directory_name = 'results/'+current_date
if not os.path.isdir(directory_name):
    os.mkdir(directory_name)
    
directory_name

'results/20260507'

In [112]:
directory_name = 'results/20251026'

In [3]:
remove_slopp = pd.read_excel('RochmanSpectra_ToRemove.xlsx', sheet_name="SLOPPe Remove")
mIRageLibraySpectra_Removed = pd.read_csv('../gloves_contamination/mIRageLibraySpectra_Removed.csv')
mIRageLibraySpectra_Removed.iloc[-1,2] = 'PE_15'

dict_slopp_reb = {'PE':'Polyethylene', 
                  'PP':'Polypropylene', 
                  'PSL':'Polystyrene'}

slopp_df = pd.read_csv('processed_data/slopp_data_raman_final_specy.csv')
del slopp_df['Unnamed: 0']
slopp_df.columns = list(np.float64(slopp_df.columns[:-4])) + list(slopp_df.columns[-4:])

slopp_df = slopp_df.query('polymer != "Cotton" and polymer !="Dyed Cellulose"').reset_index(drop=True)
slopp_df.loc[slopp_df['polymer'] == "Polyester",'polymer'] = "Polyethylene Terephthalate"
slopp_df = slopp_df.loc[[x not in np.array(remove_slopp['Example']) for x in slopp_df['id']], :].reset_index(drop=True)

rebecca_raman_df = pd.read_csv('processed_data/rebecca_data_raman_final_specy.csv')
del rebecca_raman_df['Unnamed: 0']
rebecca_raman_df.columns = list(np.float64(rebecca_raman_df.columns[:-4])) + list(rebecca_raman_df.columns[-4:])

rebecca_raman_df = rebecca_raman_df.drop(np.where([x in list(mIRageLibraySpectra_Removed.query('Signal == "Raman"')['ID']) for x in rebecca_raman_df['id']])[0],
                             axis=0).reset_index(drop=True)

rebecca_raman_df = rebecca_raman_df.query('polymer in @dict_slopp_reb').reset_index(drop=True)
rebecca_raman_df['polymer'] = [dict_slopp_reb[x] for x in rebecca_raman_df['polymer']]

In [4]:
folders = np.array(os.listdir('processed_data'))
file_names = folders[['active' in x and 'raman_saturation' in x for x in folders]]
file_names = ['processed_data/'+x for x in file_names]
ids_sat = []
for f in file_names:
    with open(f, 'r') as f:
        saturated_wn_dict = json.load(f)

    ids_sat = ids_sat + list(saturated_wn_dict.keys())
    #ids_sat_list.append(ids_sat)
    

In [5]:
folders = np.array(os.listdir())
file_names = folders[['active_data' in x and 'raman_specy_particle' in x for x in folders]]
file_names

array([], dtype='<U48')

In [6]:
folders = np.array(os.listdir('processed_data_old'))
file_names = folders[['active_data' in x and 'raman_specy_particle' in x for x in folders]]
file_names = ['processed_data_old/'+x for x in file_names]
particle_info_list = []

for f in file_names:
    particle_info = pd.read_csv(f)
    particle_info= particle_info.loc[:,['id','size','area','perimeter','circularity','distance']]
    #del active_data_raman['Unnamed: 0']
    #active_data_raman.columns = list(np.float64(active_data_raman.columns[:-7])) + list(active_data_raman.columns[-7:])

    #active_data_raman['polymer'] = ''
    #active_data_raman['label'] = ''

    particle_info_list.append(particle_info)
    
particle_info = pd.concat(particle_info_list, axis=0).reset_index(drop=True)
particle_info#.to_csv('particle_info_active_data.csv')

,id,size,area,perimeter,circularity,distance
0,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,2.387170,4.554399,8.426622,0.805787,0.347270
1,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,2.920689,6.603032,11.091790,0.674274,0.232162
2,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,2.390522,4.571330,9.162588,0.684073,0.381362
3,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,1.626374,2.133287,6.326505,0.669604,0.190898
4,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,2.835288,6.179760,11.504420,0.586596,0.358090
...,...,...,...,...,...,...
3316,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_130,0.971786,0.732929,3.272318,0.859788,0.182585
3317,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_131,1.090201,0.957468,3.770599,0.845950,0.371000
3318,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_132,0.872364,0.591004,3.126375,0.759536,0.332471
3319,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_133,0.902293,0.603713,2.996222,0.844741,0.252571


In [11]:
folders = np.array(os.listdir('processed_data'))
file_names = folders[['active_data' in x and 'raman_specy.' in x for x in folders]]
file_names = ['processed_data/'+x for x in file_names]
active_data_raman_list = []

for f in file_names:
    active_data_raman = pd.read_csv(f)
    del active_data_raman['Unnamed: 0']
    active_data_raman.columns = list(np.float64(active_data_raman.columns[:-7])) + list(active_data_raman.columns[-7:])

    active_data_raman['polymer'] = ''
    active_data_raman['label'] = ''

    active_data_raman_list.append(active_data_raman)
    
active_data_raman = pd.concat(active_data_raman_list, axis=0).reset_index(drop=True)
ids_no_signal = np.array(active_data_raman.query('no_signal>0')['id'])
del active_data_raman['wn_saturated']
del active_data_raman['no_signal']
del active_data_raman['id_file']

wn_raman = np.float64(slopp_df.columns[:-4])

min_raman_1 = 600
max_raman_1 = 1000

min_raman_2 = 1600
max_raman_2 = 3500

idx_1 = list(np.logical_or(
    np.logical_and(
        wn_raman > min_raman_1,
        wn_raman < max_raman_1),
    np.logical_and(
        wn_raman > min_raman_2,
        wn_raman < max_raman_2))) + [True]*4

min_raman_1 = 600
max_raman_1 = 910

min_raman_2 = 1040
max_raman_2 = 2310

min_raman_3 = 2500
max_raman_3 = 3500

idx_1 = list(
    np.logical_or( 
        np.logical_or(
            np.logical_and(
                wn_raman > min_raman_1,
                wn_raman < max_raman_1),
            np.logical_and(
                wn_raman > min_raman_2,
                wn_raman < max_raman_2)
            ),   
        np.logical_and(
            wn_raman > min_raman_3,
            wn_raman < max_raman_3)
    )
    ) + [True]*4

idx_2 = idx_1[:-4] + [False]*4

slopp_df = slopp_df.loc[:,idx_1]
rebecca_raman_df = rebecca_raman_df.loc[:,idx_1]
active_data_raman = active_data_raman.loc[:,idx_1+[True]]

slopp_df.iloc[:,:-4] = (slopp_df.iloc[:,:-4] - slopp_df.iloc[:,:-4].mean(1).to_numpy()[:,None]) / slopp_df.iloc[:,:-4].std(1).to_numpy()[:,None]
rebecca_raman_df.iloc[:,:-4] = (rebecca_raman_df.iloc[:,:-4] - rebecca_raman_df.iloc[:,:-4].mean(1).to_numpy()[:,None]) / rebecca_raman_df.iloc[:,:-4].std(1).to_numpy()[:,None]
active_data_raman.iloc[:,:-5] = (active_data_raman.iloc[:,:-5] - active_data_raman.iloc[:,:-5].mean(1).to_numpy()[:,None]) / active_data_raman.iloc[:,:-5].std(1).to_numpy()[:,None]


#ids_no_signal = np.concatenate([
#        ids_no_signal, 
#        np.array([
#            'MEC_04_014_DC_Active_20240808_S2_GridSi_20241210_91'
#        ])])

ids_no_signal = np.unique(
    np.concatenate([ids_no_signal,
    np.array(active_data_raman.loc[active_data_raman.iloc[:,:-5].T.std()<.01,'id'])]))

idx_raman_nan = np.unique(np.where(np.isnan(active_data_raman.iloc[:,:-5]))[0])


In [119]:
#active_data_raman['saturation'] = [x in ids_sat for x in active_data_raman['id']]
#active_data_raman['no_signal'] = [x in ids_no_signal for x in active_data_raman['id']]
#active_data_raman.to_csv(directory_name+'/active_data_raman_specy.csv')
active_data_raman.to_csv(directory_name+'/active_data_raman_specy.csv')

In [12]:
remove_flopp = pd.read_excel('RochmanSpectra_ToRemove.xlsx', sheet_name="FLOPPe Remove")

flopp_df = pd.read_csv('processed_data/flopp_data_ir_final.csv')
del flopp_df['Unnamed: 0']
flopp_df.columns = list(np.float64(flopp_df.columns[:-4])) + list(flopp_df.columns[-4:])
flopp_df = flopp_df.query('polymer != "Cotton"').reset_index(drop=True)
flopp_df.columns = list(np.float64(flopp_df.columns[:-4])) + list(flopp_df.columns[-4:])
flopp_df = flopp_df.loc[[x not in np.array(remove_flopp['Example']) for x in flopp_df['id']], :].reset_index(drop=True)

rebecca_ir_df = pd.read_csv('processed_data/rebecca_data_ir_final.csv')
del rebecca_ir_df['Unnamed: 0']
rebecca_ir_df.columns = list(np.float64(rebecca_ir_df.columns[:-4])) + list(rebecca_ir_df.columns[-4:])

rebecca_ir_df = rebecca_ir_df.drop(np.where([x in list(mIRageLibraySpectra_Removed.query('Signal == "PTIR"')['ID']) for x in rebecca_ir_df['id']])[0],
                             axis=0).reset_index(drop=True)
rebecca_ir_df = rebecca_ir_df.query('polymer in @dict_slopp_reb').reset_index(drop=True)
rebecca_ir_df['polymer'] = [dict_slopp_reb[x] for x in rebecca_ir_df['polymer']]

In [13]:
folders = np.array(os.listdir('processed_data'))
file_names = folders[['active_data' in x and 'ir.' in x for x in folders]]
file_names = ['processed_data/'+x for x in file_names]
active_data_ir_list = []

for f in file_names:
    if f == 'active_data_all_ir.csv':
        continue
    active_data_ir = pd.read_csv(f)
    del active_data_ir['Unnamed: 0']
    active_data_ir.columns = list(np.float64(active_data_ir.columns[:-4])) + list(active_data_ir.columns[-4:])

    active_data_ir['polymer'] = ''
    active_data_ir['label'] = ''

    active_data_ir_list.append(active_data_ir)
    
active_data_ir = pd.concat(active_data_ir_list, axis=0).reset_index(drop=True)
background_ids = np.array(active_data_ir.query('background == "background"')['id'])

flopp_df.iloc[:,:-4] = (flopp_df.iloc[:,:-4] - flopp_df.iloc[:,:-4].mean(1).to_numpy()[:,None]) / flopp_df.iloc[:,:-4].std(1).to_numpy()[:,None]
rebecca_ir_df.iloc[:,:-4] = (rebecca_ir_df.iloc[:,:-4] - rebecca_ir_df.iloc[:,:-4].mean(1).to_numpy()[:,None]) / rebecca_ir_df.iloc[:,:-4].std(1).to_numpy()[:,None]
active_data_ir.iloc[:,:-6] = (active_data_ir.iloc[:,:-6] - active_data_ir.iloc[:,:-6].mean(1).to_numpy()[:,None]) / active_data_ir.iloc[:,:-6].std(1).to_numpy()[:,None]


In [124]:
active_data_ir.to_csv(directory_name+'/active_data_ir.csv')

In [9]:
active_data_ir = active_data_ir.loc[["ICB" not in x for x in active_data_ir['id']],:].reset_index(drop=True)
active_data_raman = active_data_raman.loc[["ICB" not in x for x in active_data_raman['id']],:].reset_index(drop=True)


In [29]:
particle_info_active_data = pd.read_csv('particle_info_active_data.csv')
del particle_info_active_data['Unnamed: 0']

In [32]:

active_data_ir = active_data_ir.merge(particle_info_active_data, on='id', how='left').iloc[:,-11:].drop(['polymer', 'label', 'background'], axis=1)


In [14]:
removed_files = [
    'MEC_02_119_DC_Sp_S6_SquareAlSi_20240502', 
    'MEC_04_014_DC_Active_20240808_S4_GridSi_20241008',  
    'MEC_04_014_DC_Active_20240808_S6_ThinSi_20241008']


In [26]:
idx_not_rem = [(1-1*any([y in x for y in removed_files]))>0 for x in active_data_ir['id']]

In [34]:
np.unique([x.split('_')[3] for x in active_data_ir['id']])

array(['DC'], dtype='<U2')

In [58]:
months = ['January', 'February', 'March', 'MarApr', 'April', 'May','June', 'July', 'August', 'September', 'October', 'November', 'December']
months2 = np.array(['January', 'February', 'March', 'March', 'April', 'May','June', 'July', 'August', 'September', 'October', 'November', 'December'])
months_dics = {
    'MEC_02_119':'April', 
    'MEC_04_014':'August', 
    'MEC_04_023':'December'
}
np.array(active_data_ir['id'])[np.array([np.sum([y in x for y in months]) for x in active_data_ir['id']])!=1]#.groupby('school').count()['id']

#active_data_ir['month'] = [months2[[y in x for y in months]][0] for x in active_data_ir['id']]


array(['MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_1',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_2',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_3', ...,
       'MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_132',
       'MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_133',
       'MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_134'], dtype=object)

In [89]:
active_data_ir['month'] = [months_dics[list(months_dics.keys())[np.where([y in x for y in months_dics.keys()])[0][0]]] for x in active_data_ir['id']]

In [105]:
schools = ['S1', 'S2', 'S3', 'S4', 'S6']

In [106]:
active_data_ir['school'] = [x.split('_')[6] if x.split('_')[6] in schools else x.split('_')[5]
                            for x in active_data_ir['id']]

In [107]:
np.unique(active_data_ir['school'])

array(['S1', 'S2', 'S3', 'S4', 'S6'], dtype=object)

In [99]:
list(active_data_ir.query('month == "April"')['id'])

['MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_1',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_2',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_3',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_4',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_5',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_6',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_7',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_8',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_9',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_10',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_11',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_12',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_13',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_14',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_15',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_16',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_17',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_18',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_19',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_20',
 'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_21',
 

In [108]:
active_data_ir

,981.0,983.0,985.0,987.0,989.0,991.0,993.0,995.0,997.0,999.0,...,2967.0,2969.0,map_x,map_y,id,background,polymer,label,month,school
0,-0.784513,-0.584427,-0.615400,-0.668458,-0.427184,-0.167760,-0.145926,-0.027152,0.113043,0.127117,...,-1.575771,-1.803649,9691.349609,-3860.500000,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,no_background,,,December,S2
1,-0.053495,-0.023477,0.031816,0.135731,0.244412,0.325810,0.415460,0.522054,0.656807,0.841321,...,-0.596654,-0.610508,9717.500000,-3835.699951,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,no_background,,,December,S2
2,-0.627206,-0.573258,-0.521066,-0.504676,-0.522112,-0.542733,-0.513320,-0.491967,-0.442517,-0.371536,...,-0.818396,-0.949653,9627.950195,-3799.750000,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,no_background,,,December,S2
3,-0.548676,1.272581,1.219814,0.370910,0.059258,-0.517378,-0.242490,0.336142,0.483606,0.526868,...,1.765397,5.333936,9667.950195,-3773.000000,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,background,,,December,S2
4,-1.230937,-1.216147,-1.109879,-1.109838,-1.161672,-1.323753,-1.407334,-1.324721,-1.215485,-1.062650,...,-0.029356,0.103452,9613.250000,-3759.800049,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,no_background,,,December,S2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3316,-1.233485,-1.098165,-1.112937,-1.164575,-0.869571,-0.502550,0.160113,0.423133,0.159831,0.093519,...,0.146535,-0.625999,18857.199219,213.500000,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_130,no_background,,,April,S6
3317,0.899339,0.872076,0.307239,-0.614345,0.021980,1.001136,0.203499,-0.130537,-0.107647,-0.279007,...,-4.604075,-6.791733,18852.800781,214.750000,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_131,no_background,,,April,S6
3318,1.323036,0.694924,-0.732937,-0.826802,0.120935,0.564522,0.200922,0.389144,0.832391,0.740790,...,-3.644772,-4.231739,18875.300781,214.850006,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_132,no_background,,,April,S6
3319,2.833750,2.344585,-0.265426,-1.566094,-1.035945,0.659355,0.917741,-0.378427,-0.516819,-0.308869,...,-3.831361,-6.771967,18860.849609,215.250000,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_133,background,,,April,S6


In [14]:

#active_data_ir_paired = active_data_ir.loc[idx_not_rem,:].query('id not in @ids_sat and id not in @background_ids and id not in @ids_no_signal').reset_index(drop=True)
#active_data_ir_unpaired_ram = active_data_ir.loc[idx_not_rem,:].query('(id not in @ids_sat and id not in @ids_no_signal) and id in @background_ids').reset_index(drop=True)
#active_data_ir_unpaired_ir = active_data_ir.loc[idx_not_rem,:].query('(id in @ids_sat or id in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
#active_data_ir_nopair = active_data_ir.loc[idx_not_rem,:].query('(id in @ids_sat or id in @ids_no_signal) and id in @background_ids').reset_index(drop=True)

active_data_ir_paired = active_data_ir.loc[:,:].query('id not in @ids_sat and id not in @background_ids and id not in @ids_no_signal').reset_index(drop=True)
active_data_ir_unpaired_ram = active_data_ir.loc[:,:].query('(id not in @ids_sat and id not in @ids_no_signal) and id in @background_ids').reset_index(drop=True)
active_data_ir_unpaired_ir = active_data_ir.loc[:,:].query('(id in @ids_sat or id in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
active_data_ir_nopair = active_data_ir.loc[:,:].query('(id in @ids_sat or id in @ids_no_signal) and id in @background_ids').reset_index(drop=True)


active_data_raman_paired = active_data_raman.query('(id not in @ids_sat and id not in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
active_data_raman_unpaired_ram = active_data_raman.query('(id not in @ids_sat and id not in @ids_no_signal) and id in @background_ids').reset_index(drop=True)
active_data_raman_unpaired_ir = active_data_raman.query('(id in @ids_sat or id in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
active_data_raman_nopair = active_data_raman.query('(id in @ids_sat or id in @ids_no_signal) and id in @background_ids').reset_index(drop=True)

In [29]:
active_data_ir_unpaired_ram['quality'] = 'unpaired Raman spectra'
active_data_ir_unpaired_ir['quality'] = 'unpaired IR spectra'
active_data_ir_nopair['quality'] = 'no pair spectra'
active_data_ir_paired['quality'] = 'paired spectra'


In [35]:
directory_name = 'results/20251114'

In [47]:
ids_ff_info = pd.concat([
    active_data_ir_unpaired_ram,
    active_data_ir_unpaired_ir,
    active_data_ir_nopair,
    active_data_ir_paired], axis=0).reset_index(drop=True).loc[np.where(np.isnan(pd.concat([
    active_data_ir_unpaired_ram,
    active_data_ir_unpaired_ir,
    active_data_ir_nopair,
    active_data_ir_paired], axis=0).reset_index(drop=True)['size']))[0],'id'].to_numpy()#.to_csv(directory_name+'/feature_finder_info_active.csv')


In [78]:
directory_name = 'results/20251217'

In [79]:
table_mp_count = pd.concat([active_data_ir_paired.groupby(['month']).count()['id'].reset_index(drop=True),
active_data_ir_unpaired_ram.groupby(['month']).count()['id'].reset_index(drop=True),
active_data_ir_unpaired_ir.groupby(['month']).count()['id'].reset_index(drop=True),
active_data_ir_nopair.groupby(['month']).count()['id'].reset_index(drop=True)], axis=1)

table_mp_count.columns = [
    'unpaired Raman spectra',
    'unpaired IR spectra',
    'no pair spectra',
    'paired spectra',
]
table_mp_count['month'] = ['April', 'August', 'December']
table_mp_count.iloc[:,[4,0,1,2,3]].to_csv(directory_name + '/active_count.csv')

In [115]:
pd.concat([pd.pivot(active_data_ir_paired.groupby(['school', 'month']).count()['id'].reset_index(),
         index='month',
         columns='school',
         values='id'),
pd.pivot(active_data_ir_unpaired_ram.groupby(['school', 'month']).count()['id'].reset_index(),
         index='month',
         columns='school',
         values='id'),
pd.pivot(active_data_ir_unpaired_ir.groupby(['school', 'month']).count()['id'].reset_index(),
         index='month',
         columns='school',
         values='id'),
pd.pivot(active_data_ir_nopair.groupby(['school', 'month']).count()['id'].reset_index(),
         index='month',
         columns='school',
         values='id')], axis=1).to_csv(directory_name+'/active_count.csv')

In [ ]:
'paired spectra',
'unpaired Raman spectra',
'unpaired IR spectra',
'no pair spectra'

In [112]:
directory_name

'results/20251217'

In [74]:
active_data_ir_paired.groupby(['month']).count()['id'].reset_index()

,month,id
0,April,391
1,August,337
2,December,512


In [15]:
print('unpaired Raman spectra: ', active_data_ir_unpaired_ram.shape[0])
print('unpaired IR spectra: ', active_data_ir_unpaired_ir.shape[0])
print('no pair spectra: ', active_data_ir_nopair.shape[0])
print('paired spectra: ', active_data_ir_paired.shape[0])

unpaired Raman spectra:  491
unpaired IR spectra:  1401
no pair spectra:  65
paired spectra:  1364


In [28]:
print('unpaired Raman spectra: ', active_data_ir_unpaired_ram.shape[0])
print('unpaired IR spectra: ', active_data_ir_unpaired_ir.shape[0])
print('no pair spectra: ', active_data_ir_nopair.shape[0])
print('paired spectra: ', active_data_ir_paired.shape[0])

unpaired Raman spectra:  491
unpaired IR spectra:  1401
no pair spectra:  65
paired spectra:  1364


In [66]:
print('unpaired Raman spectra: ', active_data_ir_unpaired_ram.shape[0])
print('unpaired IR spectra: ', active_data_ir_unpaired_ir.shape[0])
print('no pair spectra: ', active_data_ir_nopair.shape[0])
print('paired spectra: ', active_data_ir_paired.shape[0])

unpaired Raman spectra:  389
unpaired IR spectra:  1354
no pair spectra:  39
paired spectra:  1240


In [80]:
print('unpaired Raman spectra: ', active_data_ir_unpaired_ram.shape[0])
print('unpaired IR spectra: ', active_data_ir_unpaired_ir.shape[0])
print('no pair spectra: ', active_data_ir_nopair.shape[0])
print('paired spectra: ', active_data_ir_paired.shape[0])

unpaired Raman spectra:  491
unpaired IR spectra:  1401
no pair spectra:  65
paired spectra:  1364


In [16]:
active_data_ir_all = pd.concat([active_data_ir_paired,
                                active_data_ir_unpaired_ram,
                                active_data_ir_unpaired_ir,
                                active_data_ir_nopair], axis=0).reset_index(drop=True)
active_data_ir_all['pair'] = (
    ['paired']*active_data_ir_paired.shape[0]+
    ['unpaired_ram']*active_data_ir_unpaired_ram.shape[0]+
    ['unpaired_ir']*active_data_ir_unpaired_ir.shape[0]+
    ['nopair']*active_data_ir_nopair.shape[0]
)
#active_data_ir_all.to_csv('active_data_all_ir.csv')

In [17]:
active_data_raman_all = pd.concat([active_data_raman_paired,
                                active_data_raman_unpaired_ram,
                                active_data_raman_unpaired_ir,
                                active_data_raman_nopair], axis=0).reset_index(drop=True)
active_data_raman_all['pair'] = (
    ['paired']*active_data_raman_paired.shape[0]+
    ['unpaired_ram']*active_data_raman_unpaired_ram.shape[0]+
    ['unpaired_ir']*active_data_raman_unpaired_ir.shape[0]+
    ['nopair']*active_data_raman_nopair.shape[0]
)
#active_data_raman_all.to_csv('active_data_all_raman.csv')

In [18]:
flopp_slopp_labels = np.array(slopp_df.groupby('polymer').count().iloc[:,-1:].merge(flopp_df.groupby('polymer').count().iloc[:,-1:], 
                                                      left_index=True,
                                                      right_index=True, how='inner').index)

flopp_df = flopp_df.query('polymer in @flopp_slopp_labels').reset_index(drop=True)
slopp_df = slopp_df.query('polymer in @flopp_slopp_labels').reset_index(drop=True)

slopp_df.groupby('polymer').count().iloc[:,-1:].merge(flopp_df.groupby('polymer').count().iloc[:,-1:], 
                                                      left_index=True,
                                                      right_index=True, how='outer')

,id_x,id_y
polymer,,
Acrylic,8,5
Acrylonitrile Butadiene Styrene,10,14
Cellulose Acetate,3,2
Polyamide,13,18
Polycarbonate,8,8
Polyethylene,43,65
Polyethylene Terephthalate,32,47
Polyethylene Vinyl Acetate,2,13
Polymethyl Methacrylate,3,2


In [19]:
idx_both_rebecca = np.array(rebecca_ir_df.loc[[x in np.array(rebecca_raman_df['id']) for x in rebecca_ir_df['id']],'id'])# 
rebecca_ir_df = rebecca_ir_df.query('id in @idx_both_rebecca').reset_index(drop=True)
rebecca_raman_df = rebecca_raman_df.query('id in @idx_both_rebecca').reset_index(drop=True)


In [20]:

(reference_library_raman, 
 calibration_set_raman, 
 test_set_raman, 
 label_table_raman) = gen_data_sets(slopp_df, 
                              rebecca_raman_df, 
                              active_data_raman_paired)

(X_train_raman, X_cal_raman, X_test_raman, 
 y_train_raman, y_cal_raman, y_test_raman, 
 label_train_raman, label_cal_raman, label_test_raman, 
 n_test_raman, wl_list_raman) = process_data(reference_library_raman,
                                             calibration_set_raman,
                                             test_set_raman)



(reference_library_ir, 
 calibration_set_ir, 
 test_set_ir, 
 label_table_ir) = gen_data_sets_ir(flopp_df, 
                              rebecca_ir_df, 
                              active_data_ir_paired)

(X_train_ir, X_cal_ir, X_test_ir, 
 y_train_ir, y_cal_ir, y_test_ir, 
 label_train_ir, label_cal_ir, label_test_ir, 
 n_test_ir, wl_list_ir) = process_data_ir(reference_library_ir,
                                 calibration_set_ir,
                                 test_set_ir)

In [86]:
pred_slopp_df = corr_matching(np.array(slopp_df.iloc[:,:-4]), 
                              slopp_df['polymer'], 
                              np.array(calibration_set_raman.iloc[:,3:]), r_min=0)

pred_flopp_df = corr_matching(np.array(flopp_df.iloc[:,:-4]), 
                              flopp_df['polymer'], 
                              np.array(calibration_set_ir.iloc[:,3:]), r_min=0)

(np.mean(np.array(pred_slopp_df[0]) == np.array(calibration_set_raman['polymer'])),
 np.mean(np.array(pred_flopp_df[0]) == np.array(calibration_set_ir['polymer'])))

(0.5219512195121951, 0.6292682926829268)

In [87]:
pred_slopp_df = corr_matching(np.array(slopp_df.iloc[:,:-4]), 
                              slopp_df['polymer'], 
                              np.array(test_set_raman.iloc[:,3:]), r_min=0)

pred_flopp_df = corr_matching(np.array(flopp_df.iloc[:,:-4]), 
                              flopp_df['polymer'], 
                              np.array(test_set_ir.iloc[:,3:]), r_min=0)

In [22]:
alpha = 0.1

In [23]:
def smx_log(X_train, X_test, labels_train, C=0.01, return_weights=False):
    
    clf = LogisticRegression(random_state=0, C=C).fit(X_train, labels_train)
    smx = clf.predict_proba(X_test)
    label_pred = clf.predict(X_test)
    if return_weights:
        return smx, clf
    
    return smx

In [24]:
# Calculate HQI for calibration set
smx_cal_ir = smx_nn(X_train_ir, X_cal_ir, label_train_ir)

# Calculate HQI for test set
smx_test_ir = smx_nn(X_train_ir, X_test_ir, label_train_ir)


smx_test_all_ir = smx_nn_all(
        X_train_ir, 
        X_test_ir, 
        label_train_ir
)

pred_flopp_df = [
    np.array(label_table_ir.iloc[np.argmax(smx_test_ir, 1),1]),
    np.max(smx_test_ir, 1),
    np.array(reference_library_ir.iloc[np.argmax(smx_test_all_ir, 0),0])
]

# Set confidence level 1 - alpha
n = smx_cal_ir.shape[0]
#alpha = 0.1
#for alpha in [alpha]:
prediction_sets_ir, qhat_ir = conform_pred(smx_cal_ir, smx_test_ir, label_cal_ir, n, alpha, cond=0)

prediction_sets_n_ir = [list(label_table_ir.loc[ps,'polymer']) for ps in prediction_sets_ir]
prediction_sets_size_ir = np.array([len(ps) for ps in prediction_sets_n_ir])

zero_label_ir = np.where(prediction_sets_size_ir == 0)[0]
single_label_ir = np.where(prediction_sets_size_ir == 1)[0]
mult_label_ir = np.where(prediction_sets_size_ir > 1)[0]

print('Sets with zero labels: ' + str(zero_label_ir.shape[0]))
print('Sets with one label: ' + str(single_label_ir.shape[0]))
print('Sets with >1 labels: ' + str(mult_label_ir.shape[0]))
print('Mean set size:', prediction_sets_ir.sum(1).mean())
print('Std set size:', prediction_sets_ir.sum(1).std())
print('HQI cutt off:', 1-qhat_ir)

Sets with zero labels: 0
Sets with one label: 226
Sets with >1 labels: 1138
Mean set size: 4.620967741935484
Std set size: 3.7374462718978387
HQI cutt off: 0.8938658598510795


In [93]:
# Calculate HQI for calibration set
smx_cal_raman = smx_log(X_train_raman, 
                  X_cal_raman, 
                  label_train_raman, C=0.002)

# Calculate HQI for test set
smx_test_raman = smx_log(X_train_raman, 
                   X_test_raman, 
                   label_train_raman, C=0.002)

#smx_test_all = smx_nn_all(
#        X_train_raman, 
#        X_test_raman, 
#        label_train_raman
#)

#pred_slopp_df = [
#    np.array(label_table_raman.iloc[np.argmax(smx_test, 1),1]),
#    np.max(smx_test, 1),
#    np.array(reference_library_raman.iloc[np.argmax(smx_test_all, 0),0])
#]

# Set confidence level 1 - alpha
n = smx_cal_raman.shape[0]
#for alpha in [alpha]:
prediction_sets_raman, qhat_raman = conform_pred(smx_cal_raman, smx_test_raman, label_cal_raman, n, alpha, cond=0)

prediction_sets_n_raman = [list(label_table_raman.loc[ps,'polymer']) for ps in prediction_sets_raman]
prediction_sets_size_raman = np.array([len(ps) for ps in prediction_sets_n_raman])

zero_label_raman = np.where(prediction_sets_size_raman == 0)[0]
single_label_raman = np.where(prediction_sets_size_raman == 1)[0]
mult_label_raman = np.where(prediction_sets_size_raman > 1)[0]

print('Sets with zero labels: ' + str(zero_label_raman.shape[0]))
print('Sets with one label: ' + str(single_label_raman.shape[0]))
print('Sets with >1 labels: ' + str(mult_label_raman.shape[0]))
print('Mean set size:', prediction_sets_raman.sum(1).mean())
print('Std set size:', prediction_sets_raman.sum(1).std())
print('HQI cutt off:', 1-qhat_raman)

Sets with zero labels: 0
Sets with one label: 444
Sets with >1 labels: 920
Mean set size: 3.0425219941348973
Std set size: 1.6899595086900332
HQI cutt off: 0.09032758588127421


In [26]:
prediction_sets = prediction_sets_ir + prediction_sets_raman
prediction_sets_n = [list(label_table_raman.loc[ps,'polymer']) for ps in prediction_sets]

In [27]:
print('Mean set size:', prediction_sets.sum(1).mean())
print('Std set size:', prediction_sets.sum(1).std())

Mean set size: 6.281524926686217
Std set size: 3.1668988165840255


In [36]:
pred_sets = pd.DataFrame({'pred_set':prediction_sets_n})
pred_sets['id'] = test_set_ir['id']
pred_sets.to_csv(directory_name+'/CP_results_active_paired.csv')

In [28]:
arg_sort_hqi_ir = np.argsort(1-smx_test_ir, 1)

output_df_ir = pd.DataFrame(
    np.array(
        label_table_ir.iloc[np.argsort(1-smx_test_ir, 1).flatten(), 1]
    ).reshape(smx_test_ir.shape)
)

output_df_ir.columns = ['top_' + str(i+1) for i in range(14)]

output_df_hqi = pd.DataFrame(1-np.sort(1-smx_test_ir, 1))
output_df_hqi.columns = ['top_' + str(i+1) +'_hqi' for i in range(14)]


for i in range(smx_test_ir.shape[0]):
    #ps_size = prediction_sets.sum(1)[i]
    #output_df_ir.iloc[i, ps_size:14] = ''
    #output_df_ir.iloc[i, 14+ps_size:28] = ''
    
    idx_hqi_ir = np.array([x in prediction_sets_n[i] for j, x in enumerate(list(label_table_ir['polymer']))])[arg_sort_hqi_ir[i,:]]
    output_df_ir.loc[i, ~idx_hqi_ir] = ''
    output_df_hqi.loc[i, ~idx_hqi_ir] = ''
    

output_df_ir = pd.concat([output_df_ir, output_df_hqi], axis=1)

for i in np.unique(label_train_ir):
    poly = label_table_ir.iloc[i, 1]
    idx_label = label_train_ir == i
    idx_top_by_label = np.argmax(smx_test_all_ir[idx_label,:], 0)
    ref_id_list = np.array(reference_library_ir.loc[idx_label,:].iloc[idx_top_by_label,0])
    output_df_ir['id_top_match_flopp_'+poly] = [x if poly in prediction_sets_n[j] else '' for j, x in enumerate(ref_id_list)]

output_df_ir['id'] = test_set_ir['id']

 #= pd.read_excel('evaluation_data/Passive_MVCP_Review.csv').to_numpy().flatten()

output_df_ir#.query('id in @Passive_RamanOnly_Manual_LikelyMPs').to_csv(directory_name+'/CP_results_passive_unpaired_raman_LikelyMPs.csv')


,top_1,top_2,top_3,top_4,top_5,top_6,top_7,top_8,top_9,top_10,...,id_top_match_flopp_Polyethylene,id_top_match_flopp_Polyethylene Terephthalate,id_top_match_flopp_Polyethylene Vinyl Acetate,id_top_match_flopp_Polymethyl Methacrylate,id_top_match_flopp_Polypropylene,id_top_match_flopp_Polystyrene,id_top_match_flopp_Polyurethane,id_top_match_flopp_Polyvinyl Chloride,id_top_match_flopp_Rubber,id
0,Polymethyl Methacrylate,,Rubber,,,,,,Polypropylene,,...,,,,PMMA 1. Orange Fragment.CSV,PP 6. Red Fragment.CSV,,,,R 8. White Bike Tire Fragment.CSV,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...
1,Polystyrene,Polyurethane,,,,Polyvinyl Chloride,,,,,...,,,,,,PS 8. White Foam.CSV,PU 2. Orange Foam.CSV,PVC 23. Pink Yoga Mat Foam.CSV,,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...
2,Polypropylene,Polystyrene,Polyurethane,,,,,,,,...,,,,,PP 38. Blue Fragment.CSV,PS 2. White Foam.CSV,PU 2. Orange Foam.CSV,,,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...
3,Polypropylene,Polystyrene,Polyethylene Vinyl Acetate,Polyamide,Acrylonitrile Butadiene Styrene,Acrylic,Polyethylene,Rubber,Polymethyl Methacrylate,,...,PE 18. White Fragment.CSV,,PEVA 11. Purple Fiber.CSV,PMMA 2. Red Fiber.CSV,PP 23. Yellow Fragment.CSV,PS 13. Pink Spoon Fragment.CSV,,,R 2. Black Bike Handle Cover Foam.CSV,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...
4,Polymethyl Methacrylate,Rubber,Polyurethane,,,,Polypropylene,,,,...,,,,PMMA 1. Orange Fragment.CSV,PP 64. Blue Fiber.CSV,,PU 7. Yellow Packaging Foam.CSV,,R 8. White Bike Tire Fragment.CSV,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1359,Polyethylene Vinyl Acetate,Polycarbonate,Polyurethane,Cellulose Acetate,Polyvinyl Chloride,Polyethylene Terephthalate,Polypropylene,Acrylic,Polymethyl Methacrylate,Polyethylene,...,PE 49. Pink Sphere.CSV,PET 14. White Microsphere.CSV,PEVA 10. Yellow Fiber.CSV,PMMA 1. Orange Fragment.CSV,PP 21. White Ribbon Fiber.CSV,,PU 20. Green 3D Printer Fragment.CSV,PVC 7. Clear Food Container Fragment.CSV,R 17. Grey Rubber Paving.CSV,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_122
1360,Polypropylene,Rubber,Polyethylene,Polyethylene Vinyl Acetate,Polycarbonate,,,,,,...,PE 27. White Fragment.CSV,,PEVA 11. Purple Fiber.CSV,,PP 4. Clear Food Container Fragment.CSV,,,,R 2. Black Bike Handle Cover Foam.CSV,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_123
1361,Rubber,Polypropylene,,,,,Polyethylene Terephthalate,,,,...,,PET 4. White Fragment.CSV,,,PP 43. Green Fragment.CSV,,,,R 3. White Shoe Sole Fragment.CSV,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_130
1362,Polystyrene,Rubber,Polypropylene,Polyurethane,Polyethylene,Polyamide,Acrylonitrile Butadiene Styrene,Cellulose Acetate,Polyvinyl Chloride,Polyethylene Terephthalate,...,PE 34. Black Fragment.CSV,PET 10. Purple Fiber Bundle.CSV,EVA 3. Grey Foam.CSV,,PP 43. Green Fragment.CSV,PS 6. White Foam.CSV,PU 2. Orange Foam.CSV,PVC 22. Blue Slinky Toy Fragment.CSV,R 6. Black Bike Tire Fragment.CSV,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_131


In [29]:
arg_sort_hqi_raman = np.argsort(1-smx_test_raman, 1)

output_df_raman = pd.DataFrame(
    np.array(
        label_table_raman.iloc[np.argsort(1-smx_test_raman, 1).flatten(), 1]
    ).reshape(smx_test_raman.shape)
)

output_df_raman.columns = ['top_' + str(i+1) for i in range(14)]

output_df_hqi = pd.DataFrame(1-np.sort(1-smx_test_raman, 1))
output_df_hqi.columns = ['top_' + str(i+1) +'_hqi' for i in range(14)]

for i in range(smx_test_raman.shape[0]):
    #ps_size = prediction_sets.sum(1)[i]
    #output_df_raman.iloc[i, ps_size:14] = ''
    #output_df_raman.iloc[i, 14+ps_size:28] = ''
    
    idx_hqi_raman = np.array([x in prediction_sets_n[i] for j, x in enumerate(list(label_table_raman['polymer']))])[arg_sort_hqi_raman[i,:]]
    output_df_raman.loc[i, ~idx_hqi_raman] = ''
    output_df_hqi.loc[i, ~idx_hqi_raman] = ''
    
output_df_raman = pd.concat([output_df_raman, output_df_hqi], axis=1)

#for i in np.unique(label_train_raman):
#    poly = label_table_raman.iloc[i, 1]
#    idx_label = label_train_raman == i
#    idx_top_by_label = np.argmax(smx_test_all[idx_label,:], 0)
#    ref_id_list = np.array(reference_library_raman.loc[idx_label,:].iloc[idx_top_by_label,0])
#    output_df_raman['id_top_match_flopp_'+poly] = [x if poly in prediction_sets_n[j] else '' for j, x in enumerate(ref_id_list)]

output_df_raman['id'] = test_set_ir['id']

 #= pd.read_excel('evaluation_data/Passive_MVCP_Review.csv').to_numpy().flatten()

output_df_raman#.query('id in @Passive_RamanOnly_Manual_LikelyMPs').to_csv(directory_name+'/CP_results_passive_unpaired_raman_LikelyMPs.csv')


,top_1,top_2,top_3,top_4,top_5,top_6,top_7,top_8,top_9,top_10,...,top_6_hqi,top_7_hqi,top_8_hqi,top_9_hqi,top_10_hqi,top_11_hqi,top_12_hqi,top_13_hqi,top_14_hqi,id
0,Polypropylene,Rubber,,,,,,,Polymethyl Methacrylate,,...,,,,0.045032,,,,,,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...
1,Polyvinyl Chloride,Polystyrene,,,,,,,Polyurethane,,...,,,,0.053792,,,,,,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...
2,Polypropylene,,,,,Polystyrene,,,,,...,0.03457,,,,,,0.008389,,,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...
3,Polypropylene,,,Polymethyl Methacrylate,Polystyrene,,Polyethylene,Acrylic,,Rubber,...,,0.026398,0.015952,,0.012292,0.011713,,0.008182,0.005428,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...
4,Polypropylene,,,,Polymethyl Methacrylate,,,Rubber,,,...,,,0.015404,,,,,0.009608,,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1359,Polyethylene Terephthalate,Polycarbonate,Polyurethane,,Polypropylene,Cellulose Acetate,Rubber,,Polymethyl Methacrylate,Polyvinyl Chloride,...,0.009531,0.00951,,0.007728,0.006715,0.00584,,0.005098,0.002564,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_122
1360,Polypropylene,Polycarbonate,,,,,Polyethylene,,,,...,,0.021115,,,,0.008933,,,0.004682,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_123
1361,Acrylic,Polypropylene,Polycarbonate,Polyethylene Terephthalate,,,,,,,...,,,,,,0.028153,,,,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_130
1362,Polypropylene,,,Polystyrene,Polyvinyl Chloride,Polyethylene Terephthalate,Polyethylene,Polyurethane,Cellulose Acetate,Rubber,...,0.032082,0.016766,0.013298,0.012527,0.010229,0.009939,,0.007547,0.005493,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_131


In [97]:
output_df_ir.iloc[0, :]

top_1                                                                           Polymethyl Methacrylate
top_2                                                                                                  
top_3                                                                                            Rubber
top_4                                                                                                  
top_5                                                                                                  
top_6                                                                                                  
top_7                                                                                                  
top_8                                                                                                  
top_9                                                                                     Polypropylene
top_10                                                          

In [98]:
output_df_raman.iloc[0, :]

top_1                                             Polypropylene
top_2                                                    Rubber
top_3                                                          
top_4                                                          
top_5                                                          
top_6                                                          
top_7                                                          
top_8                                                          
top_9                                   Polymethyl Methacrylate
top_10                                                         
top_11                                                         
top_12                                                         
top_13                                                         
top_14                                                         
top_1_hqi                                              0.297592
top_2_hqi                               

In [103]:
Active_MVCP_Review = pd.read_csv('evaluation_data/Active_MVCP_Review.csv')['ID'].to_numpy().flatten()
output_df_raman.query('id in @Active_MVCP_Review').to_csv(directory_name+'/CP_results_active_paired_raman_LikelyMPs.csv')
output_df_ir.query('id in @Active_MVCP_Review').to_csv(directory_name+'/CP_results_active_paired_ir_LikelyMPs.csv')


In [52]:
smx_all_IR = 1-np.power(smx_nn(X_train_ir, X_cal_ir, label_train_ir),1)
smx_all_Raman = 1-np.power(smx_nn(X_train_raman, X_cal_raman, label_train_raman),1)

smx_all_IR_test = 1-np.power(smx_nn(X_train_ir, X_test_ir, label_train_ir),1)
smx_all_Raman_test = 1-np.power(smx_nn(X_train_raman, X_test_raman, label_train_raman),1)


score_list = [smx_all_IR, smx_all_Raman]
score_list_test = [smx_all_IR_test, smx_all_Raman_test]

#alpha = .1
seed = 1
K = 2
quant_smx_list = []
rest_smx_list = []
cal_smx_list = []
val_smx_list = []
U_list = gen_u(K, seed=0)

for i in range(K):
    #quant_smx, rest_smx, quant_labels, rest_labels, idx_quant = split_cal_test(score_list[i], label_ir, cal_prop=0.2, seed=seed)
    quant_smx, cal_smx, quant_labels, cal_labels, idx_quant = split_cal_test(score_list[i], label_cal_ir, cal_prop=0.1, seed=seed)
    
    val_smx, val_labels = score_list_test[i], None
    
    quant_smx_list.append(quant_smx)
    #rest_smx_list.append(rest_smx)
    cal_smx_list.append(cal_smx)
    val_smx_list.append(val_smx)

n = cal_labels.shape[0]

prediction_sets = mvcp_2(quant_smx_list,       
                         quant_labels,       
                         cal_smx_list,
                         cal_labels,  
                         val_smx_list,
                         alpha,     
                         val_labels,                 
                         U=U_list[0])

prediction_sets_n_mvcp = [list(label_table_raman.loc[ps,'polymer']) for ps in prediction_sets]
prediction_sets_size = np.array([len(ps) for ps in prediction_sets_n_mvcp])

zero_label = np.where(prediction_sets_size == 0)[0]
single_label = np.where(prediction_sets_size == 1)[0]
mult_label = np.where(prediction_sets_size > 1)[0]

print('Sets with zero labels: ' + str(zero_label.shape[0]))
print('Sets with one label: ' + str(single_label.shape[0]))
print('Sets with >1 labels: ' + str(mult_label.shape[0]))
print('Mean set size:', prediction_sets.sum(1).mean())
print('Std set size:', prediction_sets.sum(1).std())

Sets with zero labels: 11
Sets with one label: 73
Sets with >1 labels: 1280
Mean set size: 8.336510263929618
Std set size: 4.7575023164419346


In [180]:
output = [prediction_sets_n[i][0] if i in single_label else 
              '--outlier--' if i in zero_label else 
              '--manual check!--' for i in range(label_test_ir.shape[0])]
output_df = pd.DataFrame({'id':test_set_ir['id'],
                          'prection_set_ir':prediction_sets_n,
                          'prection_set_raman':prediction_sets_n_raman,
                          'prection_set_mvcp':prediction_sets_n_mvcp,
                              #'output':output
                              })

output_df['polymer_flopp'] = pred_flopp_df[0]#np.array(pred_flopp_df[0])#['polymer_ref']
output_df['hqi_flopp'] = pred_flopp_df[1]#['hqi_ref']
output_df['id_top_match_flopp'] = pred_flopp_df[2]#np.array(flopp_df.iloc[np.array(pred_flopp_df[0].index),-1])

output_df['polymer_slopp'] = pred_slopp_df[0]#np.array(pred_slopp_df[0])#['polymer_ref']
output_df['hqi_slopp'] = pred_slopp_df[1]#['hqi_ref']
output_df['id_top_match_slopp'] = pred_slopp_df[2]#np.array(slopp_df.iloc[np.array(pred_slopp_df[0].index),-1])


output_df.to_csv(directory_name+'/CP_results_active_paired.csv')

# Unpaired Raman

In [104]:
(reference_library_raman, 
 calibration_set_raman, 
 test_set_raman, 
 label_table_raman) = gen_data_sets(slopp_df, 
                              rebecca_raman_df, 
                              active_data_raman_unpaired_ram)

(X_train_raman, X_cal_raman, X_test_raman, 
 y_train_raman, y_cal_raman, y_test_raman, 
 label_train_raman, label_cal_raman, label_test_raman, 
 n_test_raman, wl_list_raman) = process_data(reference_library_raman,
                                 calibration_set_raman,
                                 test_set_raman)


(reference_library_ir, 
 calibration_set_ir, 
 test_set_ir, 
 label_table_ir) = gen_data_sets_ir(flopp_df, 
                              rebecca_ir_df, 
                              active_data_ir_unpaired_ram)

(X_train_ir, X_cal_ir, X_test_ir, 
 y_train_ir, y_cal_ir, y_test_ir, 
 label_train_ir, label_cal_ir, label_test_ir, 
 n_test_ir, wl_list_ir) = process_data_ir(reference_library_ir,
                                 calibration_set_ir,
                                 test_set_ir)



        
pred_slopp_df = corr_matching(np.array(slopp_df.iloc[:,:-4]), 
                              slopp_df['polymer'], 
                              np.array(test_set_raman.iloc[:,3:]), r_min=0)

pred_flopp_df = corr_matching(np.array(flopp_df.iloc[:,:-4]), 
                              flopp_df['polymer'], 
                              np.array(test_set_ir.iloc[:,3:]), r_min=0)


In [105]:
# Calculate HQI for calibration set
smx_cal = smx_log(X_train_raman, 
                 X_cal_raman, 
                 label_train_raman, C=.002)

# Calculate HQI for test set
smx_test = smx_log(X_train_raman, 
                  X_test_raman, 
                  label_train_raman, C=.002)

#smx_test_all = smx_nn_all(
#        X_train_raman, 
#        X_test_raman, 
#        label_train_raman
#)

#pred_slopp_df = [
#    np.array(label_table_raman.iloc[np.argmax(smx_test, 1),1]),
#    np.max(smx_test, 1),
#    np.array(reference_library_raman.iloc[np.argmax(smx_test_all, 0),0])
#]

# Set confidence level 1 - alpha
n = smx_cal.shape[0]
alpha = 0.1
#for alpha in [alpha]:
prediction_sets, qhat = conform_pred(smx_cal, smx_test, label_cal_raman, n, alpha, cond=0)

prediction_sets_n_raman = [list(label_table_raman.loc[ps,'polymer']) for ps in prediction_sets]
prediction_sets_size = np.array([len(ps) for ps in prediction_sets_n_raman])

zero_label = np.where(prediction_sets_size == 0)[0]
single_label = np.where(prediction_sets_size == 1)[0]
mult_label = np.where(prediction_sets_size > 1)[0]

print('Sets with zero labels: ' + str(zero_label.shape[0]))
print('Sets with one label: ' + str(single_label.shape[0]))
print('Sets with >1 labels: ' + str(mult_label.shape[0]))
print('Mean set size:', prediction_sets.sum(1).mean())
print('Std set size:', prediction_sets.sum(1).std())

Sets with zero labels: 0
Sets with one label: 206
Sets with >1 labels: 285
Mean set size: 2.745417515274949
Std set size: 1.7144273324427663


In [107]:
output_df = pd.DataFrame(
    np.array(
        label_table_raman.iloc[np.argsort(1-smx_test, 1).flatten(), 1]
    ).reshape(smx_test.shape)
)

output_df.columns = ['top_' + str(i+1) for i in range(14)]

output_df_hqi = pd.DataFrame(1-np.sort(1-smx_test, 1))
output_df_hqi.columns = ['top_' + str(i+1) +'_hqi' for i in range(14)]

output_df = pd.concat([output_df, output_df_hqi], axis=1)

for i in range(smx_test.shape[0]):
    ps_size = prediction_sets.sum(1)[i]
    output_df.iloc[i, ps_size:14] = ''
    output_df.iloc[i, 14+ps_size:28] = ''

#for i in np.unique(label_train_raman):
#    poly = label_table_raman.iloc[i, 1]
#    idx_label = label_train_raman == i
#    idx_top_by_label = np.argmax(smx_test_all[idx_label,:], 0)
#    ref_id_list = np.array(reference_library_raman.loc[idx_label,:].iloc[idx_top_by_label,0])
#    output_df['id_top_match_flopp_'+poly] = [x if poly in prediction_sets_n[j] else '' for j, x in enumerate(ref_id_list)]

output_df['id'] = test_set_ir['id']

Active_Raman_Review = pd.read_csv('evaluation_data/Active_Raman_Review.csv')['ID'].to_numpy().flatten()

output_df.query('id in @Active_Raman_Review').to_csv(directory_name+'/CP_results_active_unpaired_raman_LikelyMPs.csv')


In [187]:
output_df = pd.DataFrame({'id':test_set_ir['id'],
                          #'prection_set_ir':prediction_sets_n,
                          'prection_set_raman':prediction_sets_n_raman,
                          #'prection_set_mvcp':prediction_sets_n_mvcp,
                              #'output':output
                              })

#output_df['polymer_flopp'] = np.array(pred_flopp_df[0])#['polymer_ref']
#output_df['hqi_flopp'] = pred_flopp_df[1]#['hqi_ref']
#output_df['id_top_match_flopp'] = np.array(flopp_df.iloc[np.array(pred_flopp_df[0].index),-1])

output_df['polymer_slopp'] = pred_slopp_df[0]#np.array(pred_slopp_df[0])#['polymer_ref']
output_df['hqi_slopp'] = pred_slopp_df[1]#['hqi_ref']
output_df['id_top_match_slopp'] = pred_slopp_df[2]#np.array(slopp_df.iloc[np.array(pred_slopp_df[0].index),-1])


output_df.to_csv(directory_name+'/CP_results_active_unpaired_raman.csv')

# Unpaired IR

In [37]:
(reference_library_raman, 
 calibration_set_raman, 
 test_set_raman, 
 label_table_raman) = gen_data_sets(slopp_df, 
                              rebecca_raman_df, 
                              active_data_raman_unpaired_ir)

(X_train_raman, X_cal_raman, X_test_raman, 
 y_train_raman, y_cal_raman, y_test_raman, 
 label_train_raman, label_cal_raman, label_test_raman, 
 n_test_raman, wl_list_raman) = process_data(reference_library_raman,
                                 calibration_set_raman,
                                 test_set_raman)


(reference_library_ir, 
 calibration_set_ir, 
 test_set_ir, 
 label_table_ir) = gen_data_sets_ir(flopp_df, 
                              rebecca_ir_df, 
                              active_data_ir_unpaired_ir)

(X_train_ir, X_cal_ir, X_test_ir, 
 y_train_ir, y_cal_ir, y_test_ir, 
 label_train_ir, label_cal_ir, label_test_ir, 
 n_test_ir, wl_list_ir) = process_data_ir(reference_library_ir,
                                 calibration_set_ir,
                                 test_set_ir)



        
pred_slopp_df = corr_matching(np.array(slopp_df.iloc[:,:-4]), 
                              slopp_df['polymer'], 
                              np.array(test_set_raman.iloc[:,3:]), r_min=0)

pred_flopp_df = corr_matching(np.array(flopp_df.iloc[:,:-4]), 
                              flopp_df['polymer'], 
                              np.array(test_set_ir.iloc[:,3:]), r_min=0)


In [38]:
# Calculate HQI for calibration set
smx_cal = smx_nn(X_train_ir, X_cal_ir, label_train_ir)

# Calculate HQI for test set
smx_test = smx_nn(X_train_ir, X_test_ir, label_train_ir)

smx_test_all = smx_nn_all(
        X_train_ir, 
        X_test_ir, 
        label_train_ir
)

pred_flopp_df = [
    np.array(label_table_ir.iloc[np.argmax(smx_test, 1),1]),
    np.max(smx_test, 1),
    np.array(reference_library_ir.iloc[np.argmax(smx_test_all, 0),0])
]

# Set confidence level 1 - alpha
n = smx_cal.shape[0]
alpha = 0.1
#for alpha in [alpha]:
prediction_sets, qhat = conform_pred(smx_cal, smx_test, label_cal_ir, n, alpha, cond=0)

prediction_sets_n = [list(label_table_ir.loc[ps,'polymer']) for ps in prediction_sets]
prediction_sets_size = np.array([len(ps) for ps in prediction_sets_n])

zero_label = np.where(prediction_sets_size == 0)[0]
single_label = np.where(prediction_sets_size == 1)[0]
mult_label = np.where(prediction_sets_size > 1)[0]

print('Sets with zero labels: ' + str(zero_label.shape[0]))
print('Sets with one label: ' + str(single_label.shape[0]))
print('Sets with >1 labels: ' + str(mult_label.shape[0]))
print('Mean set size:', prediction_sets.sum(1).mean())
print('Std set size:', prediction_sets.sum(1).std())

Sets with zero labels: 0
Sets with one label: 472
Sets with >1 labels: 929
Mean set size: 2.4875089221984297
Std set size: 1.7866247863362201


In [40]:
pred_sets = pd.DataFrame({'pred_set':prediction_sets_n})
pred_sets['id'] = test_set_ir['id']
pred_sets.to_csv(directory_name+'/CP_results_active_onlyIR.csv')

In [110]:
output_df = pd.DataFrame(np.array(label_table_ir.iloc[np.argsort(1-smx_test, 1).flatten(), 1]).reshape(smx_test.shape))
output_df.columns = ['top_' + str(i+1) for i in range(14)]

output_df_hqi = pd.DataFrame(1-np.sort(1-smx_test, 1))
output_df_hqi.columns = ['top_' + str(i+1) +'_hqi' for i in range(14)]

output_df = pd.concat([output_df, output_df_hqi], axis=1)

for i in range(smx_test.shape[0]):
    ps_size = prediction_sets.sum(1)[i]
    output_df.iloc[i, ps_size:14] = ''
    output_df.iloc[i, 14+ps_size:28] = ''

for i in np.unique(label_train_ir):
    poly = label_table_ir.iloc[i, 1]
    idx_label = label_train_ir == i
    idx_top_by_label = np.argmax(smx_test_all[idx_label,:], 0)
    ref_id_list = np.array(reference_library_ir.loc[idx_label,:].iloc[idx_top_by_label,0])
    output_df['id_top_match_flopp_'+poly] = [x if poly in prediction_sets_n[j] else '' for j, x in enumerate(ref_id_list)]

output_df['id'] = test_set_ir['id']

Active_PTIR_Review = pd.read_csv('evaluation_data/Active_PTIR_Review.csv')['id'].to_numpy().flatten()

output_df.query('id in @Active_PTIR_Review').to_csv(directory_name+'/CP_results_active_unpaired_ir_LikelyMPs.csv')


In [192]:
output_df = pd.DataFrame({'id':test_set_ir['id'],
                          'prection_set_ir':prediction_sets_n,
                          #'prection_set_raman':prediction_sets_n_raman,
                          #'prection_set_mvcp':prediction_sets_n_mvcp,
                              #'output':output
                              })

output_df['polymer_flopp'] = pred_flopp_df[0]#np.array(pred_flopp_df[0])#['polymer_ref']
output_df['hqi_flopp'] = pred_flopp_df[1]#['hqi_ref']
output_df['id_top_match_flopp'] = pred_flopp_df[2]#np.array(flopp_df.iloc[np.array(pred_flopp_df[0].index),-1])

#output_df['polymer_slopp'] = np.array(pred_slopp_df[0])#['polymer_ref']
#output_df['hqi_slopp'] = pred_slopp_df[1]#['hqi_ref']
#output_df['id_top_match_slopp'] = np.array(slopp_df.iloc[np.array(pred_slopp_df[0].index),-1])


output_df.to_csv(directory_name+'/CP_results_active_unpaired_ir.csv')